In [1]:
import duckdb
from pathlib import Path

RAW = Path("../data/raw/2023")
con = duckdb.connect()
for tabla in ["main", "awards", "awards_suppliers", "contracts"]:
    ruta = (RAW / f"{tabla}.csv").as_posix()
    con.sql(f"CREATE VIEW {tabla} AS SELECT * FROM read_csv('{ruta}', all_varchar = true)")

con.sql("""
    SELECT 'main' AS tabla, count(*) AS filas FROM main
    UNION ALL SELECT 'awards', count(*) FROM awards
    UNION ALL SELECT 'awards_suppliers', count(*) FROM awards_suppliers
    UNION ALL SELECT 'contracts', count(*) FROM contracts
""")

┌──────────────────┬───────┐
│      tabla       │ filas │
│     varchar      │ int64 │
├──────────────────┼───────┤
│ main             │ 46116 │
│ awards           │ 36982 │
│ awards_suppliers │ 36971 │
│ contracts        │ 38338 │
└──────────────────┴───────┘

In [2]:
campos = [
    "buyer_id", "tender_procurementMethodDetails", "tender_value_amount_PEN",
    "tender_numberOfTenderers", "tender_tenderPeriod_startDate",
    "tender_enquiryPeriod_durationInDays",
]
con.sql("SELECT " + ", ".join(
    f"round(100.0 * count(nullif(trim({c}), '')) / count(*), 1) AS {c}" for c in campos
) + " FROM main")

┌──────────┬─────────────────────────────────┬─────────────────────────┬──────────────────────────┬───────────────────────────────┬─────────────────────────────────────┐
│ buyer_id │ tender_procurementMethodDetails │ tender_value_amount_PEN │ tender_numberOfTenderers │ tender_tenderPeriod_startDate │ tender_enquiryPeriod_durationInDays │
│  double  │             double              │         double          │          double          │            double             │               double                │
├──────────┼─────────────────────────────────┼─────────────────────────┼──────────────────────────┼───────────────────────────────┼─────────────────────────────────────┤
│    100.0 │                           100.0 │                   100.0 │                     90.8 │                          92.3 │                                63.3 │
└──────────┴─────────────────────────────────┴─────────────────────────┴──────────────────────────┴───────────────────────────────┴───────────────────

In [3]:
con.sql("""
    SELECT substr(tender_datePublished, 1, 4) AS anio, count(*) AS procesos
    FROM main GROUP BY 1 ORDER BY 1
""")

┌─────────┬──────────┐
│  anio   │ procesos │
│ varchar │  int64   │
├─────────┼──────────┤
│ 2014    │        2 │
│ 2015    │     1481 │
│ 2016    │     2074 │
│ 2017    │        4 │
│ 2018    │        8 │
│ 2019    │       24 │
│ 2020    │       80 │
│ 2021    │      270 │
│ 2022    │     1950 │
│ 2023    │    40223 │
└─────────┴──────────┘
  10 rows  2 columns

In [4]:
con.sql("""
    SELECT date_diff('day',
               CAST(substr(tender_tenderPeriod_startDate, 1, 10) AS DATE),
               CAST(substr(tender_tenderPeriod_endDate, 1, 10) AS DATE)) AS dias,
           count(*) AS procesos
    FROM main
    WHERE tender_tenderPeriod_startDate <> ''
    GROUP BY 1
""")

┌───────┬──────────┐
│ dias  │ procesos │
│ int64 │  int64   │
├───────┼──────────┤
│     0 │    42563 │
└───────┴──────────┘

In [5]:
con.sql("""
    SELECT tender_procurementMethodDetails AS metodo,
           count(*) FILTER (WHERE tender_numberOfTenderers = '1') AS un_participante,
           count(*) AS procesos,
           round(100.0 * un_participante / procesos, 1) AS pct
    FROM main GROUP BY 1 ORDER BY procesos DESC LIMIT 6
""")

┌─────────────────────────────┬─────────────────┬──────────┬────────┐
│           metodo            │ un_participante │ procesos │  pct   │
│           varchar           │      int64      │  int64   │ double │
├─────────────────────────────┼─────────────────┼──────────┼────────┤
│ Adjudicación Simplificada   │             625 │    24469 │    2.6 │
│ Subasta Inversa Electrónica │             146 │     6259 │    2.3 │
│ Comparación de Precios      │              41 │     3383 │    1.2 │
│ Contratación Directa        │            2869 │     3319 │   86.4 │
│ Licitación Pública          │              13 │     1853 │    0.7 │
│ Concurso Público            │              16 │     1376 │    1.2 │
└─────────────────────────────┴─────────────────┴──────────┴────────┘

In [6]:
con.sql("""
    SELECT tender_procurementMethodDetails AS metodo,
           median(CAST(tender_enquiryPeriod_durationInDays AS INTEGER)) AS mediana_dias,
           count(*) AS procesos
    FROM main
    WHERE tender_enquiryPeriod_durationInDays <> ''
    GROUP BY 1 ORDER BY procesos DESC LIMIT 6
""")

┌──────────────────────────────────────────┬──────────────┬──────────┐
│                  metodo                  │ mediana_dias │ procesos │
│                 varchar                  │    double    │  int64   │
├──────────────────────────────────────────┼──────────────┼──────────┤
│ Adjudicación Simplificada                │          2.0 │    24461 │
│ Licitación Pública                       │         14.0 │     1813 │
│ Concurso Público                         │         13.0 │     1349 │
│ Adjudicación Directa Selectiva           │          2.0 │      413 │
│ Procedimiento Especial de Contratación   │          1.0 │      244 │
│ Adjudicación Simplificada - Ley N° 31728 │          4.0 │      218 │
└──────────────────────────────────────────┴──────────────┴──────────┘

In [7]:
con.sql("""
    WITH a AS (
        SELECT main_ocid, sum(CAST(value_amount AS DOUBLE)) AS adjudicado, count(*) AS n
        FROM awards GROUP BY 1
    )
    SELECT count(*) FILTER (WHERE ratio BETWEEN 0.995 AND 1.005) AS igual_al_referencial,
           count(*) AS procesos,
           round(median(ratio), 3) AS mediana
    FROM (
        SELECT a.adjudicado / nullif(CAST(m.tender_value_amount AS DOUBLE), 0) AS ratio
        FROM a JOIN main m ON m.ocid = a.main_ocid
        WHERE a.n = 1
    )
""")

┌──────────────────────┬──────────┬─────────┐
│ igual_al_referencial │ procesos │ mediana │
│        int64         │  int64   │ double  │
├──────────────────────┼──────────┼─────────┤
│                14997 │    33956 │   0.992 │
└──────────────────────┴──────────┴─────────┘

In [8]:
con.sql("""
    SELECT count(*) FILTER (WHERE implementation_finalValue_amount <> '') AS con_valor_final,
           count(*) FILTER (WHERE CAST(nullif(implementation_finalValue_amount, '') AS DOUBLE)
                            > CAST(value_amount AS DOUBLE) * 1.0001) AS final_mayor,
           count(*) AS contratos
    FROM contracts
""")

┌─────────────────┬─────────────┬───────────┐
│ con_valor_final │ final_mayor │ contratos │
│      int64      │    int64    │   int64   │
├─────────────────┼─────────────┼───────────┤
│            2400 │        1542 │     38338 │
└─────────────────┴─────────────┴───────────┘